# View streamlined MIMIC experiment results

Load previously saved experiment artifacts and display the analysis tables and figures. This notebook does not fit models or rerun experimental conditions.


In [ ]:
# Set a name explicitly, or use None to view the most recently updated experiment.
EXPERIMENT_NAME = None

# Optional overrides; leave as None to use the profile and default artifact directory.
CONFIG_PATH = None
ARTIFACT_DIR = None


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "MIMIC" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

EXPERIMENT_ROOT = PROJECT_ROOT / "manuscript" / "experiments"
for path in [PROJECT_ROOT / "src", EXPERIMENT_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 240)

from streamlined.config import (
    ensure_artifact_dirs,
    load_config,
    resolve_experiment_artifact_dir,
)
from streamlined.runner import artifact_manifest, run_profile
from streamlined.plotting import (
    generate_critical_difference_diagram,
    generate_mean_learning_curves,
    plot_learning_curves,
    plot_real_equivalent_by_dataset,
)


In [ ]:
config_path = Path(CONFIG_PATH) if CONFIG_PATH else EXPERIMENT_ROOT / "configs" / "full.yaml"
artifact_base_dir = ARTIFACT_DIR or EXPERIMENT_ROOT / "artifacts"
artifact_dir = resolve_experiment_artifact_dir(artifact_base_dir, EXPERIMENT_NAME)
config = load_config(config_path, artifact_dir=artifact_dir)
ensure_artifact_dirs(config)

print(f"Experiment: {artifact_dir.name}")
print(f"Artifact directory: {config.artifact_root}")
display(artifact_manifest(config)[["artifact", "relative_path", "exists"]])


In [ ]:
tables = run_profile(config, run_experiment=False)
display(tables["learning_curves"].head())
display(tables["aulc"].head())
display(tables["pairwise"].head())
display(tables["real_equivalence_summary"].head())
display(tables["real_equivalence"].head())
display(tables["regime"].head())


In [ ]:
if not tables["learning_curves"].empty:
    print("Higher ROC AUC is better")
    for ratio in sorted(tables["learning_curves"]["imbalance_ratio"].unique()):
        print("-" * 120)
        print(f"\n=== Imbalance ratio {ratio:g}:1 ===")
        fig, ax = generate_mean_learning_curves(tables["learning_curves"], imbalance_ratio=ratio)
        display(fig)
        plt.close(fig)
        for dataset_key in sorted(tables["learning_curves"]["dataset_key"].unique()):
            fig, ax = plot_learning_curves(
                tables["learning_curves"], dataset_key=dataset_key, imbalance_ratio=ratio
            )
            display(fig)
            plt.close(fig)
        ratio_equivalence = tables["real_equivalence"].loc[
            tables["real_equivalence"]["imbalance_ratio"].eq(ratio)
        ]
        if not ratio_equivalence.empty:
            fig, ax = plot_real_equivalent_by_dataset(ratio_equivalence, imbalance_ratio=ratio)
            display(fig)
            plt.close(fig)

if not tables["aulc"].empty:
    print("Lower rank is better")
    for segment in ["full", "early"]:
        fig, ax = generate_critical_difference_diagram(tables["aulc"], segment=segment)
        display(fig)
        plt.close(fig)


## Results analysis

Across all imbalance ratios, the real-balanced baseline achieves the highest mean full-curve AULC, while the four synthetic-balancing methods become progressively less competitive as imbalance increases. Direct displacement is the strongest synthetic method at every ratio, with mean AULC declining from 0.849 at 2:1 to 0.826 at 10:1, compared with 0.857 to 0.862 for real-balanced data; the other synthetic methods follow closely at mild imbalance but separate more clearly at 5:1 and 10:1, with latent SMOTE showing the largest decline. The real-equivalent analysis reinforces this pattern: averaged over available dataset and training-size comparisons, synthetic data corresponds to roughly 66% of the real sample size at 2:1, 49% at 3:1, 33% at 5:1, and 20% at 10:1. Equivalent fractions at the higher ratios should be interpreted cautiously because interpolation is only possible when a synthetic score falls within the observed real-data learning curve, leaving fewer valid comparisons—at 10:1, values are available for only two datasets. Overall, synthetic balancing recovers a substantial share of the value of additional real observations under mild imbalance, but its effective replacement value falls sharply as class imbalance becomes more severe.
